# Out-of-Distribution (OOD) Evaluation

This notebook evaluates the pre-trained GRU model (trained on the Sarcasm News Headline dataset) on a curated set of out-of-distribution examples from `ood/train1.json` and `ood/train2.json`.

In [14]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from models import SentimentRNN
from preprocess import collate_fn

In [15]:
# Configuration
TOKENIZER_PATH = "tokenizer_bpe.json"
OOD_FILES = ["ood/train1.json", "ood/train2.json"]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters (must match the training configuration)
EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 1
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5
MODEL_TYPE = "gru"

In [16]:
# Generate paths for checkpoints (best_model + all epoch checkpoints)
MODEL_PATHS = ["checkpoints/best_model.pt"] + [f"checkpoints/model_epoch_{i}.pt" for i in range(1, 26)]

In [17]:
class OODDataset(Dataset):
    def __init__(self, json_files, tokenizer):
        self.data = []
        self.tokenizer = tokenizer
        
        label_map = {"real": 0, "fake": 1}
        
        for file_path in json_files:
            if not os.path.exists(file_path):
                continue
            with open(file_path, 'r', encoding='utf-8') as f:
                batch_data = json.load(f)
                for item in batch_data:
                    # Map labels if they are strings
                    label = item["label"]
                    if isinstance(label, str):
                        label = label_map.get(label.lower(), 0)
                    
                    self.data.append({
                        "headline": item["headline"],
                        "label": label
                    })
                    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer.encode(item["headline"])
        
        return {
            "input_ids": torch.tensor(encoding.ids, dtype=torch.long),
            "label": torch.tensor(item["label"], dtype=torch.float)
        }

In [18]:
# Load Tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f"Loaded tokenizer with vocab size: {vocab_size}")

# Load Data
ood_dataset = OODDataset(OOD_FILES, tokenizer)
ood_loader = DataLoader(ood_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
print(f"Loaded {len(ood_dataset)} OOD examples.")

Loaded tokenizer with vocab size: 30000
Loaded 100 OOD examples.


In [19]:
def evaluate_ood(model, iterator, device):
    epoch_loss = 0
    epoch_acc = 0
    criterion = nn.BCELoss()
    
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text = batch["input_ids"].to(device)
            labels = batch["label"].unsqueeze(1).to(device)
            lengths = batch["lengths"]
            
            predictions = model(text, lengths)
            loss = criterion(predictions, labels)
            
            rounded_preds = torch.round(predictions)
            correct = (rounded_preds == labels).float()
            acc = correct.sum() / len(correct)
            
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

print(f"{'Checkpoint':<30} | {'Loss':<6} | {'Accuracy':<8}")
print("-" * 50)

for model_path in MODEL_PATHS:
    if not os.path.exists(model_path):
        continue
        
    # Initialize Model
    model = SentimentRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, 
                         N_LAYERS, BIDIRECTIONAL, DROPOUT, model_type=MODEL_TYPE)
    
    # Load state dict (handling wrapped epoch checkpoints)
    checkpoint = torch.load(model_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    else:
        state_dict = checkpoint
        
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    
    loss, acc = evaluate_ood(model, ood_loader, DEVICE)
    print(f"{model_path:<30} | {loss:.3f}  | {acc*100:6.2f}%")

Checkpoint                     | Loss   | Accuracy
--------------------------------------------------
checkpoints/best_model.pt      | 2.160  |  64.84%
checkpoints/model_epoch_1.pt   | 0.689  |  65.62%
checkpoints/model_epoch_2.pt   | 0.819  |  62.50%
checkpoints/model_epoch_3.pt   | 1.012  |  64.06%
checkpoints/model_epoch_4.pt   | 1.137  |  63.28%
checkpoints/model_epoch_5.pt   | 0.910  |  64.84%
checkpoints/model_epoch_6.pt   | 0.969  |  66.41%
checkpoints/model_epoch_7.pt   | 1.024  |  63.28%
checkpoints/model_epoch_8.pt   | 1.431  |  62.50%
checkpoints/model_epoch_9.pt   | 1.397  |  62.50%
checkpoints/model_epoch_10.pt  | 1.254  |  63.28%
checkpoints/model_epoch_11.pt  | 1.536  |  62.50%
checkpoints/model_epoch_12.pt  | 1.382  |  62.50%
checkpoints/model_epoch_13.pt  | 1.660  |  62.50%
checkpoints/model_epoch_14.pt  | 1.360  |  63.28%
checkpoints/model_epoch_15.pt  | 1.921  |  64.06%
checkpoints/model_epoch_16.pt  | 1.721  |  64.06%
checkpoints/model_epoch_17.pt  | 1.821  |  63.28

In [20]:
# Final per-file analysis for the best model
best_model_path = "checkpoints/best_model.pt"
if os.path.exists(best_model_path):
    model = SentimentRNN(vocab_size, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, 
                         N_LAYERS, BIDIRECTIONAL, DROPOUT, model_type=MODEL_TYPE)
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
    model = model.to(DEVICE)
    
    print("\nDetailed Results for Best Model:")
    for ood_file in OOD_FILES:
        if not os.path.exists(ood_file):
            continue
        ds = OODDataset([ood_file], tokenizer)
        loader = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=collate_fn)
        _, file_acc = evaluate_ood(model, loader, DEVICE)
        print(f"Accuracy on {ood_file}: {file_acc*100:.2f}%")


Detailed Results for Best Model:
Accuracy on ood/train1.json: 52.60%
Accuracy on ood/train2.json: 73.44%
